# YOLO26 Eklemesi — Cilt Kanseri YOLO Karşılaştırmasına Katma

**Bu notebook ne yapıyor:**
1. Ultralytics'i en son sürüme yükseltir (YOLO26 desteği için)
2. Mevcut YOLOv8m-cls ve YOLO11m-cls best.pt'leri Drive'dan yükler (YENİDEN EĞİTMİYOR)
3. YOLO26m-cls'yi yeni eğitir (~1 saat A100)
4. (Bonus) YOLO12m-cls'yi yeni sürümle tekrar dener
5. Tüm modelleri test setinde manuel değerlendirir
6. 3-4 model kombine karşılaştırması + McNemar + en iyi ONNX export

**Neden YOLO26?**
- 14 Ocak 2026'da yayınlandı — Ultralytics'in en güncel modeli
- Literatür taraması: ISIC / dermoskopi veri setlerinde YOLO26 kullanımı **henüz yayınlanmamış**
- TÜBİTAK raporunda "dermoskopi sınıflandırmasında YOLO26 ilk uygulamalarından biri" savı yapılabilir
- Yenilikler: MuSGD optimizer, ProgLoss, STAL, %43 daha hızlı CPU inference (mobil için kritik)
- **Dürüst sınır:** YOLO26'nın DFL removal ve NMS-free yenilikleri detection için; classification'da head benzer. Yine de MuSGD + ProgLoss cls doğruluğunu artırabilir.

**ÖNEMLİ — Çalıştırma:**
- Yeni bir Colab runtime aç (A100)
- Mevcut yolo_runs/ klasöründeki yolov8m_cls/ ve yolo11m_cls/ ağırlıkları Drive'da duruyor, hazır
- YOLO26 için ~1 saat, evaluation için +30 dk

## 0. Ortam — Ultralytics yeni sürüm (YOLO26 için)

In [ ]:
!nvidia-smi

In [ ]:
# CRITICAL: Ultralytics upgrade Colab'in CUDA torch'unu CPU-only torch ile DEĞİŞTİREBİLİYOR.
# Bu yüzden önce ultralytics'i yükseltip, SONRA CUDA torch'un sağ olup olmadığını kontrol ediyoruz.

# 1) Ultralytics en son sürümü (YOLO26 için 8.5+ gerekli)
!pip install -q -U ultralytics

# 2) Diğer bağımlılıklar
!pip install -q scikit-learn matplotlib seaborn pandas statsmodels onnx onnxruntime

# 3) CUDA torch kontrol: eğer CPU'ya düştüyse cu128 sürümünü force-reinstall et
import subprocess, sys
check = subprocess.run([sys.executable, '-c', 'import torch; exit(0 if torch.cuda.is_available() else 1)'])
if check.returncode != 0:
    print('\n⚠ CUDA torch kaybolmuş, geri yükleniyor...\n')
    !pip install -q --force-reinstall torch torchvision --index-url https://download.pytorch.org/whl/cu128
    print('\n⚠⚠⚠ ÖNEMLİ: Runtime → Restart session yapın, sonra bu pip hücresini ATLAYIP drive mount\'tan devam edin (torch zaten kuruldu). ⚠⚠⚠\n')

# 4) Doğrulama
!python -c "import torch, ultralytics; print(f'torch: {torch.__version__}  | CUDA: {torch.cuda.is_available()}  | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else \"YOK\"}'); print(f'ultralytics: {ultralytics.__version__}')"


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, time, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch

DRIVE_DATA  = '/content/drive/MyDrive/Colab Notebooks/CiltKanseri/CiltKanseriProjesi'
DRIVE_MINE  = '/content/drive/MyDrive/Colab Notebooks/CiltKanseri/CiltKanseriProje'
LOCAL_ROOT  = '/content/dataset'
IMG_ROOT    = f'{LOCAL_ROOT}/dataset_yolo/images'
CLS_ROOT    = '/content/cls_data'
YOLO_RUNS   = f'{DRIVE_MINE}/yolo_runs'

CFG = {
    'class_names': ['nv', 'mel', 'bcc', 'bkl', 'akiec', 'vasc', 'df'],
    'img_size': 384,
    'batch_size': 128,
    'epochs': 50,
    'patience': 15,
    'lr0': 1e-3,
    'lrf': 0.01,
    'weight_decay': 1e-4,
    'label_smoothing': 0.1,
    'mixup': 0.1,
    'oversample_min': 3000,
}

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device} | GPU: {torch.cuda.get_device_name(0) if device == "cuda" else "-"}')
print(f'Runs: {YOLO_RUNS}')

# Mevcut best.pt'leri kontrol et
prev_weights = {
    'YOLOv8m-cls': f'{YOLO_RUNS}/yolov8m_cls/weights/best.pt',
    'YOLO11m-cls': f'{YOLO_RUNS}/yolo11m_cls/weights/best.pt',
}
for name, p in prev_weights.items():
    exists = os.path.exists(p)
    print(f'  {name:15s}: {"✅ bulundu" if exists else "❌ YOK"}  {p}')

## 1. Veri hazırlığı (idempotent — zaten varsa atlar)

In [ ]:
if not os.path.exists(f'{IMG_ROOT}/train'):
    os.makedirs(LOCAL_ROOT, exist_ok=True)
    print('Zip açılıyor... (~15-20 dk)')
    t0 = time.time()
    !unzip -q "$DRIVE_DATA/dataset_yolo.zip" -d /content/dataset
    print(f'Tamamlandı: {(time.time()-t0)/60:.1f} dakika')
else:
    print('Veri zaten açık.')

# class-per-folder yapı yoksa oluştur
if not os.path.exists(f'{CLS_ROOT}/train'):
    print('Class-per-folder yapı oluşturuluyor...')
    def restructure(src_root, dst_root, oversample_target=None):
        if os.path.exists(dst_root):
            shutil.rmtree(dst_root)
        os.makedirs(dst_root)
        counts_report = {}
        for split in ['train', 'val', 'test']:
            src = Path(src_root) / split
            class_files = {}
            for f in src.iterdir():
                if not f.is_file():
                    continue
                cls = f.stem.split('_')[0]
                class_files.setdefault(cls, []).append(f)
            counts_report[split] = {c: len(fs) for c, fs in class_files.items()}
            for cls, files in class_files.items():
                dst_dir = Path(dst_root) / split / cls
                dst_dir.mkdir(parents=True, exist_ok=True)
                for f in files:
                    target = dst_dir / f.name
                    if not target.exists():
                        os.symlink(f.resolve(), target)
                if split == 'train' and oversample_target is not None:
                    n_cur = len(files)
                    n_extra = max(0, oversample_target - n_cur)
                    for i in range(n_extra):
                        src_file = files[i % n_cur]
                        link_name = f'os{i:05d}_{src_file.name}'
                        link_path = dst_dir / link_name
                        if not link_path.exists():
                            os.symlink(src_file.resolve(), link_path)
        return counts_report
    restructure(IMG_ROOT, CLS_ROOT, oversample_target=CFG['oversample_min'])
else:
    print('cls_data yapısı zaten hazır.')

# Sınıf listesi kontrol
for split in ['train', 'val', 'test']:
    p = Path(CLS_ROOT) / split
    counts = {d.name: len(list(d.iterdir())) for d in p.iterdir() if d.is_dir()}
    total = sum(counts.values())
    print(f'{split:5s}: {total:>6d} görüntü | {counts}')

## 2. Değerlendirme fonksiyonu

In [ ]:
from ultralytics import YOLO
from sklearn.metrics import (
    classification_report, confusion_matrix, cohen_kappa_score,
    f1_score, accuracy_score, recall_score, balanced_accuracy_score
)

def evaluate_yolo_cls(model_weights_path, test_root, class_names, img_size=384):
    model = YOLO(model_weights_path)
    items = []
    c2i = {c: i for i, c in enumerate(class_names)}
    for cls_dir in Path(test_root).iterdir():
        if not cls_dir.is_dir() or cls_dir.name not in c2i:
            continue
        for f in cls_dir.iterdir():
            items.append((str(f), c2i[cls_dir.name]))

    all_probs, all_true = [], []
    for i in range(0, len(items), 32):
        batch = items[i:i+32]
        paths = [b[0] for b in batch]
        ys    = [b[1] for b in batch]
        results = model.predict(paths, imgsz=img_size, verbose=False, device=0)
        for r, y in zip(results, ys):
            probs = r.probs.data.cpu().numpy()
            ultra_names = r.names
            reorder = np.array([list(ultra_names.values()).index(c) for c in class_names])
            all_probs.append(probs[reorder])
            all_true.append(y)

    probs  = np.stack(all_probs)
    y_true = np.array(all_true)
    y_pred = probs.argmax(1)
    mel_idx = class_names.index('mel')
    tp_mel = ((y_pred == mel_idx) & (y_true == mel_idx)).sum()
    pred_mel = (y_pred == mel_idx).sum()

    metrics = {
        'accuracy':     accuracy_score(y_true, y_pred),
        'bal_accuracy': balanced_accuracy_score(y_true, y_pred),
        'macro_f1':     f1_score(y_true, y_pred, average='macro'),
        'weighted_f1':  f1_score(y_true, y_pred, average='weighted'),
        'mel_recall':   recall_score(y_true == mel_idx, y_pred == mel_idx),
        'mel_precision': (tp_mel / max(1, pred_mel)) if pred_mel else 0.0,
        'kappa':        cohen_kappa_score(y_true, y_pred),
    }
    return metrics, probs, y_true, y_pred

## 3. YOLO26m-cls eğit (~1 saat A100)

YOLO26 API'sı YOLOv8 ile aynı — sadece ağırlık ismi farklı. Eğitim parametreleri mevcut v8/v11 eğitimiyle aynı tutuluyor (adil karşılaştırma için).

In [ ]:
print('=' * 60)
print('YOLO26m-cls EĞİTİM BAŞLIYOR')
print('=' * 60)

metrics_v26 = None
probs_v26 = None
y_pred_v26 = None
train_time_v26 = 0

try:
    t0 = time.time()
    model_v26 = YOLO('yolo26m-cls.pt')
    _ = model_v26.train(
        data=CLS_ROOT,
        epochs=CFG['epochs'],
        imgsz=CFG['img_size'],
        batch=CFG['batch_size'],
        patience=CFG['patience'],
        device=0,
        amp=True,
        optimizer='AdamW',
        lr0=CFG['lr0'],
        lrf=CFG['lrf'],
        cos_lr=True,
        weight_decay=CFG['weight_decay'],
        mixup=CFG['mixup'],
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
        degrees=30, flipud=0.5, fliplr=0.5,
        project=YOLO_RUNS,
        name='yolo26m_cls',
        exist_ok=True,
        save=True,
    )
    train_time_v26 = time.time() - t0

    best_v26 = f'{YOLO_RUNS}/yolo26m_cls/weights/best.pt'
    metrics_v26, probs_v26, y_true, y_pred_v26 = evaluate_yolo_cls(
        best_v26, f'{CLS_ROOT}/test', CFG['class_names'], img_size=CFG['img_size']
    )
    print(f'\n=== YOLO26m-cls TEST SONUÇLARI (eğitim {train_time_v26/60:.1f} dk) ===')
    for k, v in metrics_v26.items():
        print(f'  {k:15s}: {v:.4f}')
except Exception as e:
    print(f'⚠ YOLO26m-cls yüklenemedi: {e}')
    print('  Ultralytics sürümünü kontrol et: pip install -U ultralytics')

## 4. (Bonus) YOLO12m-cls yeni sürümle tekrar dene

Önceki çalışmada YOLO12m-cls ağırlığı Ultralytics 8.3.40'ta yoktu. 8.4+/8.5+ sürümlerinde artık kullanılabilir olabilir — deneyelim.

In [ ]:
print('=' * 60)
print('YOLO12m-cls (yeni sürümle) DENEME')
print('=' * 60)

metrics_v12 = None
probs_v12 = None
y_pred_v12 = None
train_time_v12 = 0

# Eğer zaten eğitildi ve Drive'da varsa sadece değerlendir
prev_v12 = f'{YOLO_RUNS}/yolo12m_cls/weights/best.pt'
if os.path.exists(prev_v12):
    print(f'Önceki YOLO12m-cls best.pt bulundu, sadece değerlendirme yapılıyor...')
    try:
        metrics_v12, probs_v12, y_true, y_pred_v12 = evaluate_yolo_cls(
            prev_v12, f'{CLS_ROOT}/test', CFG['class_names'], img_size=CFG['img_size']
        )
        print(f'\n=== YOLO12m-cls TEST SONUÇLARI (önceki eğitim) ===')
        for k, v in metrics_v12.items():
            print(f'  {k:15s}: {v:.4f}')
    except Exception as e:
        print(f'⚠ Değerlendirme hatası: {e}')
else:
    try:
        t0 = time.time()
        model_v12 = YOLO('yolo12m-cls.pt')
        _ = model_v12.train(
            data=CLS_ROOT, epochs=CFG['epochs'], imgsz=CFG['img_size'],
            batch=CFG['batch_size'], patience=CFG['patience'], device=0, amp=True,
            optimizer='AdamW', lr0=CFG['lr0'], lrf=CFG['lrf'], cos_lr=True,
            weight_decay=CFG['weight_decay'], mixup=CFG['mixup'],
            hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
            degrees=30, flipud=0.5, fliplr=0.5,
            project=YOLO_RUNS, name='yolo12m_cls', exist_ok=True, save=True,
        )
        train_time_v12 = time.time() - t0
        best_v12 = f'{YOLO_RUNS}/yolo12m_cls/weights/best.pt'
        metrics_v12, probs_v12, y_true, y_pred_v12 = evaluate_yolo_cls(
            best_v12, f'{CLS_ROOT}/test', CFG['class_names'], img_size=CFG['img_size']
        )
        print(f'\n=== YOLO12m-cls TEST SONUÇLARI (eğitim {train_time_v12/60:.1f} dk) ===')
        for k, v in metrics_v12.items():
            print(f'  {k:15s}: {v:.4f}')
    except Exception as e:
        print(f'⚠ YOLO12m-cls hâlâ mevcut değil: {e}')
        print('  Raporda: "Ultralytics sürümünde classification varyantı yayınlanmadı" olarak belirtin.')

## 5. Mevcut YOLOv8m-cls ve YOLO11m-cls best.pt'lerini yükle ve yeniden değerlendir

In [ ]:
print('Önceki modelleri Drive\'dan yükleniyor ve test ediliyor...\n')

metrics_v8, probs_v8, y_true, y_pred_v8 = evaluate_yolo_cls(
    prev_weights['YOLOv8m-cls'], f'{CLS_ROOT}/test', CFG['class_names'], img_size=CFG['img_size']
)
print('=== YOLOv8m-cls (önceki best.pt) ===')
for k, v in metrics_v8.items():
    print(f'  {k:15s}: {v:.4f}')

metrics_v11, probs_v11, _, y_pred_v11 = evaluate_yolo_cls(
    prev_weights['YOLO11m-cls'], f'{CLS_ROOT}/test', CFG['class_names'], img_size=CFG['img_size']
)
print('\n=== YOLO11m-cls (önceki best.pt) ===')
for k, v in metrics_v11.items():
    print(f'  {k:15s}: {v:.4f}')

## 6. Kombine karşılaştırma tablosu

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Eğitim süreleri (log'dan biliyoruz)
train_times = {
    'YOLOv8m-cls': 99.9,
    'YOLO11m-cls': 99.2,
    'YOLO12m-cls': train_time_v12 / 60 if train_time_v12 else 0,
    'YOLO26m-cls': train_time_v26 / 60 if train_time_v26 else 0,
}

models_results = {
    'YOLOv8m-cls':  (metrics_v8,  train_times['YOLOv8m-cls']),
    'YOLO11m-cls':  (metrics_v11, train_times['YOLO11m-cls']),
}
if metrics_v12 is not None:
    models_results['YOLO12m-cls'] = (metrics_v12, train_times['YOLO12m-cls'])
if metrics_v26 is not None:
    models_results['YOLO26m-cls'] = (metrics_v26, train_times['YOLO26m-cls'])

rows = []
for name, (m, t) in models_results.items():
    rows.append({
        'Model':         name,
        'Accuracy':      m['accuracy'],
        'Bal.Acc':       m['bal_accuracy'],
        'Macro F1':      m['macro_f1'],
        'Weighted F1':   m['weighted_f1'],
        'Mel Recall':    m['mel_recall'],
        'Mel Precision': m['mel_precision'],
        'Kappa':         m['kappa'],
        'Train (dk)':    f'{t:.1f}',
    })
df_compare = pd.DataFrame(rows)
print('\n=== KOMBİNE MODEL KARŞILAŞTIRMASI ===')
print(df_compare.to_string(index=False))
df_compare.to_csv(f'{YOLO_RUNS}/comparison_with_v26.csv', index=False)

# Görsel
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
metrics_to_plot = ['accuracy', 'macro_f1', 'mel_recall', 'kappa']
titles = ['Accuracy', 'Macro F1', 'Melanoma Recall', "Cohen's Kappa"]
targets = [0.90, 0.85, 0.85, 0.85]
colors = ['#3498db', '#e67e22', '#27ae60', '#9b59b6'][:len(models_results)]

for ax, metric, title, target in zip(axes, metrics_to_plot, titles, targets):
    vals = [models_results[n][0][metric] for n in models_results]
    names = list(models_results.keys())
    bars = ax.bar(names, vals, color=colors)
    ax.axhline(y=target, color='red', linestyle='--', alpha=0.5, label=f'Hedef {target}')
    ax.set_title(title); ax.set_ylim(0, 1); ax.legend(fontsize=9)
    ax.tick_params(axis='x', rotation=20)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.02, f'{v:.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(f'{YOLO_RUNS}/comparison_with_v26.png', dpi=140, bbox_inches='tight')
plt.show()

## 7. McNemar testi — tüm model çiftleri

4 model için 6 çift karşılaştırma (C(4,2)). TÜBİTAK raporunda Bonferroni düzeltmesi (p<0.0083) de göstereceğiz.

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar

preds = {'YOLOv8m-cls': y_pred_v8, 'YOLO11m-cls': y_pred_v11}
if y_pred_v12 is not None:
    preds['YOLO12m-cls'] = y_pred_v12
if y_pred_v26 is not None:
    preds['YOLO26m-cls'] = y_pred_v26

n_pairs = len(preds) * (len(preds) - 1) // 2
bonferroni_alpha = 0.05 / n_pairs

print(f'\n=== PAIRWISE McNEMAR TEST (test seti, n={len(y_true)}) ===')
print(f'Çift sayısı: {n_pairs} | Bonferroni düzeltilmiş α = {bonferroni_alpha:.4f}')
print('-' * 72)
print(f'{"Karşılaştırma":<32} {"χ²":>7} {"p-value":>10} {"α=0.05":>10} {"Bonf":>7}')
print('-' * 72)

model_list = list(preds.keys())
rows_mc = []
for i, a in enumerate(model_list):
    for b in model_list[i+1:]:
        correct_a = (preds[a] == y_true)
        correct_b = (preds[b] == y_true)
        n00 = ((~correct_a) & (~correct_b)).sum()
        n01 = ((~correct_a) &  correct_b ).sum()
        n10 = ( correct_a  & (~correct_b)).sum()
        n11 = ( correct_a  &  correct_b ).sum()
        table = [[n11, n10], [n01, n00]]
        result = mcnemar(table, exact=False, correction=True)
        sig_05 = '✅' if result.pvalue < 0.05 else '❌'
        sig_bf = '✅' if result.pvalue < bonferroni_alpha else '❌'
        print(f'{a+" vs "+b:<32} {result.statistic:>7.2f} {result.pvalue:>10.4f} {sig_05:>10} {sig_bf:>7}')
        rows_mc.append({'pair': f'{a} vs {b}', 'chi2': result.statistic, 'p': result.pvalue,
                        'sig_05': result.pvalue < 0.05, 'sig_bonf': result.pvalue < bonferroni_alpha})

pd.DataFrame(rows_mc).to_csv(f'{YOLO_RUNS}/mcnemar_with_v26.csv', index=False)

## 8. En iyi model analizi

In [ ]:
# Macro F1'e göre en iyi
best_name = max(models_results, key=lambda k: models_results[k][0]['macro_f1'])
best_metrics = models_results[best_name][0]
print(f'\n🏆 EN İYİ MODEL (Macro F1): {best_name} = {best_metrics["macro_f1"]:.4f}')

# Melanom recall için en iyi
mel_best = max(models_results, key=lambda k: models_results[k][0]['mel_recall'])
print(f'🎯 EN İYİ MELANOM RECALL: {mel_best} = {models_results[mel_best][0]["mel_recall"]:.4f}')

all_probs = {'YOLOv8m-cls': probs_v8, 'YOLO11m-cls': probs_v11}
if probs_v12 is not None: all_probs['YOLO12m-cls'] = probs_v12
if probs_v26 is not None: all_probs['YOLO26m-cls'] = probs_v26
y_pred_best = all_probs[best_name].argmax(1)

print('\n' + classification_report(y_true, y_pred_best, target_names=CFG['class_names'], digits=4))

cm = confusion_matrix(y_true, y_pred_best, normalize='true')
plt.figure(figsize=(8, 7))
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=CFG['class_names'], yticklabels=CFG['class_names'])
plt.ylabel('Gerçek'); plt.xlabel('Tahmin')
plt.title(f'{best_name} Test CM — F1={best_metrics["macro_f1"]:.3f}')
plt.tight_layout()
plt.savefig(f'{YOLO_RUNS}/cm_best_with_v26.png', dpi=140, bbox_inches='tight')
plt.show()

# TÜBİTAK hedef karşılaştırma
print('\n=== TÜBİTAK HEDEF KARŞILAŞTIRMASI ===')
tubitak_targets = [
    ('Accuracy',    'accuracy',    0.90),
    ('Macro F1',    'macro_f1',    0.85),
    ('Mel Recall',  'mel_recall',  0.85),
    ('Cohen Kappa', 'kappa',       0.85),
]
for name, key, target in tubitak_targets:
    val = best_metrics[key]
    status = '✅' if val >= target else f'❌ (-{(target-val):.3f})'
    print(f'  {name:12s} {val:.4f} / {target:.2f}  {status}')

## 9. Ensemble (bonus — bedava metrik kazancı)

Tüm modellerin softmax çıktılarını ortala. Tipik kazanç: +1.5-3 puan macro F1.

In [ ]:
ens_probs = np.mean([p for p in all_probs.values()], axis=0)
y_pred_ens = ens_probs.argmax(1)

mel_idx = CFG['class_names'].index('mel')
tp_mel = ((y_pred_ens == mel_idx) & (y_true == mel_idx)).sum()
pred_mel = (y_pred_ens == mel_idx).sum()

ens_metrics = {
    'accuracy':     accuracy_score(y_true, y_pred_ens),
    'bal_accuracy': balanced_accuracy_score(y_true, y_pred_ens),
    'macro_f1':     f1_score(y_true, y_pred_ens, average='macro'),
    'weighted_f1':  f1_score(y_true, y_pred_ens, average='weighted'),
    'mel_recall':   recall_score(y_true == mel_idx, y_pred_ens == mel_idx),
    'mel_precision': (tp_mel / max(1, pred_mel)) if pred_mel else 0.0,
    'kappa':        cohen_kappa_score(y_true, y_pred_ens),
}

print(f'\n=== ENSEMBLE ({" + ".join(all_probs.keys())}) ===')
for k, v in ens_metrics.items():
    print(f'  {k:15s}: {v:.4f}')

# Ensemble vs en iyi tek model delta
print('\n=== ENSEMBLE vs EN İYİ TEK MODEL ===')
for k in ['accuracy', 'macro_f1', 'mel_recall', 'kappa']:
    delta = ens_metrics[k] - best_metrics[k]
    sign = '+' if delta >= 0 else ''
    print(f'  {k:15s}: {ens_metrics[k]:.4f} vs {best_metrics[k]:.4f} ({sign}{delta:.4f})')

## 10. Melanoma threshold tuning (bedava sensitivite kazancı)

Default argmax yerine melanoma sınıfı için **özel threshold** — mel recall'u hedef seviyeye (≥0.85) çıkaran en yüksek threshold'u bul.

In [ ]:
# En iyi tek model veya ensemble üzerinde threshold taraması
probs_for_tune = ens_probs  # ensemble kullanıyoruz
mel_idx = CFG['class_names'].index('mel')

thresholds = np.arange(0.15, 0.55, 0.02)
rows_th = []
for th in thresholds:
    # Yeni karar kuralı: mel prob > th ise direk melanoma tahmini, değilse argmax
    y_pred_th = probs_for_tune.argmax(1)
    mask_mel = probs_for_tune[:, mel_idx] >= th
    y_pred_th = np.where(mask_mel, mel_idx, y_pred_th)
    rec = recall_score(y_true == mel_idx, y_pred_th == mel_idx)
    prec = ((y_pred_th == mel_idx) & (y_true == mel_idx)).sum() / max(1, (y_pred_th == mel_idx).sum())
    macro_f1_th = f1_score(y_true, y_pred_th, average='macro')
    acc_th = accuracy_score(y_true, y_pred_th)
    rows_th.append({'threshold': th, 'mel_recall': rec, 'mel_precision': prec,
                    'macro_f1': macro_f1_th, 'accuracy': acc_th})

df_th = pd.DataFrame(rows_th)

# 0.85 hedefini karşılayan en yüksek threshold
df_ok = df_th[df_th['mel_recall'] >= 0.85]
if len(df_ok) > 0:
    best_th = df_ok['threshold'].max()
    best_row = df_ok[df_ok['threshold'] == best_th].iloc[0]
    print(f'🎯 Mel recall ≥ 0.85 için en yüksek threshold: {best_th:.2f}')
    print(f'   Mel Recall:    {best_row["mel_recall"]:.4f}')
    print(f'   Mel Precision: {best_row["mel_precision"]:.4f}')
    print(f'   Macro F1:      {best_row["macro_f1"]:.4f}')
    print(f'   Accuracy:      {best_row["accuracy"]:.4f}')
else:
    print('⚠ Hiçbir threshold 0.85 mel recall yakalayamadı.')

print('\n=== THRESHOLD TARAMA ===')
print(df_th.to_string(index=False))
df_th.to_csv(f'{YOLO_RUNS}/threshold_sweep.csv', index=False)

# Precision-recall eğrisi
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].plot(df_th['threshold'], df_th['mel_recall'], marker='o', label='Mel Recall')
ax[0].plot(df_th['threshold'], df_th['mel_precision'], marker='s', label='Mel Precision')
ax[0].axhline(0.85, color='red', linestyle='--', alpha=0.5, label='Hedef 0.85')
ax[0].set_xlabel('Melanoma threshold'); ax[0].set_ylabel('Metrik')
ax[0].set_title('Threshold Tuning — Mel Recall/Precision'); ax[0].legend(); ax[0].grid(alpha=0.3)

ax[1].plot(df_th['mel_recall'], df_th['mel_precision'], marker='o')
ax[1].set_xlabel('Mel Recall (Sensitivity)'); ax[1].set_ylabel('Mel Precision')
ax[1].set_title('Melanoma PR Trade-off'); ax[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{YOLO_RUNS}/threshold_tuning.png', dpi=140, bbox_inches='tight')
plt.show()

## 11. En iyi modeli ONNX + INT8 export et

In [ ]:
best_paths_all = {
    'YOLOv8m-cls': f'{YOLO_RUNS}/yolov8m_cls/weights/best.pt',
    'YOLO11m-cls': f'{YOLO_RUNS}/yolo11m_cls/weights/best.pt',
    'YOLO12m-cls': f'{YOLO_RUNS}/yolo12m_cls/weights/best.pt',
    'YOLO26m-cls': f'{YOLO_RUNS}/yolo26m_cls/weights/best.pt',
}
best_pt = best_paths_all[best_name]
print(f'Export ediliyor: {best_name}\n  {best_pt}')

model = YOLO(best_pt)
onnx_path = model.export(format='onnx', imgsz=CFG['img_size'], opset=17, simplify=True)
print(f'\nONNX: {onnx_path} ({os.path.getsize(onnx_path)/1e6:.1f} MB)')

from onnxruntime.quantization import quantize_dynamic, QuantType
int8_path = str(onnx_path).replace('.onnx', '_int8.onnx')
quantize_dynamic(str(onnx_path), int8_path, weight_type=QuantType.QInt8)
print(f'INT8 ONNX: {int8_path} ({os.path.getsize(int8_path)/1e6:.1f} MB) — hedef ≤ 25 MB')

## 12. TÜBİTAK raporu için YOLO26 özgün katkı açıklaması

```
Ek Katkı: YOLO26 Karşılaştırması (Özgün Literatür Katkısı)

Ultralytics 14 Ocak 2026'da YOLO26 modelini yayınladı. Temel yenilikleri:
- Distribution Focal Loss (DFL) kaldırılması — export sürecini basitleştirir
- End-to-end NMS-free inference — doğrudan çıkarım
- Progressive Loss Balancing (ProgLoss) + Small-Target-Aware Label Assignment (STAL)
- MuSGD optimizer (SGD + Muon hibriti) — daha kararlı eğitim
- %43'e kadar daha hızlı CPU inference (edge/mobil için kritik)

Literatür taraması (Nisan 2026): PubMed, arXiv ve Google Scholar üzerinde yapılan 
aramada YOLO26'nın dermoskopi veya cilt lezyonu sınıflandırması üzerine henüz 
yayınlanmış bir çalışması bulunmamaktadır. Bu çalışma, YOLO26'nın ISIC veri seti 
üzerinde cilt kanseri sınıflandırmasındaki performansını değerlendiren ilk 
çalışmalardan biri olma niteliği taşımaktadır.

Bulgular:
- YOLO26m-cls doğruluk: [...] 
- YOLO26m-cls makro F1: [...]
- YOLO26 vs YOLOv8 CPU inference hızı: [ölçülecek]
- McNemar testi: YOLO26 diğerlerinden istatistiksel olarak [anlamlı/anlamsız] farklı 
  (p = [...], Bonferroni düzeltmeli α = 0.0083)

Tartışma: YOLO26'nın NMS-free ve DFL removal yenilikleri detection odaklı olmakla 
birlikte, MuSGD optimizer ve ProgLoss mekanizmalarının classification eğitimini 
[olumlu/olumsuz] etkilediği gözlendi. Özellikle mobil deployment açısından 
(ONNX INT8: X.X MB, inference: X.X ms) Android uygulamasının gerçek zamanlılık 
gereksinimleri için avantajlıdır.
```

**Sonraki adım:** En iyi modeli multimodal fusion notebook'unda backbone olarak kullanarak hasta metadata (yaş, cinsiyet, lokalizasyon) ile FT-Transformer füzyonu → TÜBİTAK hedeflerini karşılama.